In [1]:
from helpers import *
from helpersmodels import *
from implementations import * 
import os

In [2]:
# Get the current directory (the project root)
PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
# Load data
x_train = np.genfromtxt(
        os.path.join(DATA_DIR, "x_train_cleaned.csv"), delimiter=",", skip_header=1
    )
y_train = np.genfromtxt(
        os.path.join(DATA_DIR, "y_train_up.csv"), delimiter=",", skip_header=1
    )
# Split
X_train, X_test, y_train, y_test=split_train(x_train,y_train)


In [3]:
#load test data
x_test_real= np.genfromtxt(
        os.path.join(DATA_DIR, "x_test_cleaned.csv"), delimiter=",", skip_header=1
    )
x_test_ids= np.genfromtxt(
        os.path.join(DATA_DIR, "x_test.csv"), delimiter=",",skip_header=1
    )
x_test_ids=x_test_ids[:, 0]

# Gradient Descent

In [8]:
# Define a list of potential learning rates (gamma values) to test
gamma_vals = [0.005, 0.01, 0.03,0.07]

# Perform k-fold cross-validation to find the best learning rate
# cross_validation_gd returns the gamma that gives the best validation performance
best_gamma_gd = cross_validation_gd(y_train, X_train, k_fold=5, gammas=gamma_vals)

# Train the model using gradient descent with the best gamma found
# mean_squared_error_gd returns the optimized weight vector and final loss
best_w_gd, _ = mean_squared_error_gd(
    y_train, X_train, np.zeros(X_train.shape[1]), max_iters=1000, gamma=best_gamma_gd
)

# Compute predicted scores (raw model outputs before applying threshold)
y_scores = X_test @ best_w_gd

# Find the best classification threshold based on F1 score
best_threshold_gd, best_f1_gd, best_accuracy_gd = optimize_threshold(y_test, y_scores)

# Generate final predictions using the optimized threshold
y_predicted = predict(y_scores, best_threshold_gd)

# Print summary of results
print("Best Gamma:", best_gamma_gd)
print("Best Threshold:", best_threshold_gd)
print("Final Accuracy:", best_accuracy_gd)
print("Final F1 Score:", best_f1_gd)

Gamma = 0.00500 | Avg F1 = 0.54048
Gamma = 0.01000 | Avg F1 = 0.54342
Gamma = 0.03000 | Avg F1 = 0.54546
Gamma = 0.07000 | Avg F1 = 0.00000
Best Gamma: 0.03
Best Threshold: -0.43958397000632177
Final Accuracy: 0.8136020501963843
Final F1 Score: 0.5495304453196002


In [ ]:
# Compute model scores on the real test set
y_test_scores = x_test_real @ best_w_gd

# Predict final labels using the optimized threshold
y_predicted = predict(y_test_scores, best_threshold_gd)

# Create the submission CSV file
create_csv_submission(x_test_ids, y_predicted, "predgd")

# Stochastic Gradient Descent

In [5]:
# Test different learning rates (gamma values) for SGD
gamma_vals = [0.0006,0.0008,0.001,0.002]

# Select best gamma using 5-fold cross-validation
best_gamma_sgd = cross_validation_sgd(y_train, X_train, k_fold=5, gammas=gamma_vals)

# Train model with best gamma using stochastic gradient descent
best_w_sgd, _ = mean_squared_error_sgd(
    y_train, X_train, np.zeros(X_train.shape[1]), max_iters=1000, gamma=best_gamma_sgd
)

# Compute prediction scores on the test set
y_scores = X_test @ best_w_sgd

# Find optimal classification threshold (maximizing F1 score)
best_threshold_sgd, best_f1_sgd, best_accuracy_sgd = optimize_threshold(y_test, y_scores)

# Predict final labels
y_predicted = predict(y_scores, best_threshold_sgd)

# Display results
print("Best Gamma:", best_gamma_sgd)
print("Best Threshold:", best_threshold_sgd)
print("Final Accuracy:", best_accuracy_sgd)
print("Final F1 Score:", best_f1_sgd)

[SGD] Gamma = 0.00060 | Avg F1 = 0.499345
[SGD] Gamma = 0.00080 | Avg F1 = 0.500449
[SGD] Gamma = 0.00100 | Avg F1 = 0.501188
[SGD] Gamma = 0.00200 | Avg F1 = 0.492646
=> Best gamma (SGD): 0.001 with Avg F1 = 0.501188
Best Gamma: 0.001
Best Threshold: -0.5678055921712923
Final Accuracy: 0.779325329396362
Final F1 Score: 0.5112290227048372


In [ ]:
# Compute scores on the real test set using the trained weights
y_test_scores = x_test_real @ best_w_sgd

# Predict labels using the best threshold found
y_predicted = predict(y_test_scores, best_threshold_sgd)

# Generate submission file for the SGD model
create_csv_submission(x_test_ids, y_predicted, "predsgd")

# Least Squares

In [3]:
# Train model using least squares
w_ls, _ = least_squares(y_train, X_train)

# Compute prediction scores on the test set
y_scores = X_test @ w_ls

# Find optimal threshold (based on F1 score)
best_threshold_ls, best_f1_ls, best_accuracy_ls = optimize_threshold(y_test, y_scores)

# Predict final labels
y_predicted = predict(y_scores, best_threshold_ls)

# Display performance metrics
print("Final Accuracy:", best_accuracy_ls)
print("Final F1 Score:", best_f1_ls)

Final Accuracy: 0.8167079862948828
Final F1 Score: 0.5507305749009969


In [6]:
# Compute scores on the real test set using least squares weights
y_test_scores = x_test_real @ w_ls

# Predict labels using the optimal threshold
y_predicted = predict(y_test_scores, best_threshold_ls)

# Generate submission file for the least squares model
create_csv_submission(x_test_ids, y_predicted, "predLS")

# Ridge Regression

In [11]:
# Test different regularization strengths (lambda values)
lambda_vals = [0.001,0.005, 0.01, 0.2]

# Select best lambda using 5-fold cross-validation
best_lambda_rr = cross_validation_ridge(y_train, X_train, k_fold=5, lambdas=lambda_vals)

# Train model with the best lambda using ridge regression
best_w_rr, _ = ridge_regression(y_train, X_train, best_lambda_rr)

# Compute prediction scores on the test set
y_scores = X_test @ best_w_rr

# Find optimal threshold (maximizing F1 score)
best_threshold_rr, best_f1_rr, best_accuracy_rr = optimize_threshold(y_test, y_scores)

# Predict final labels
y_predicted = predict(y_scores, best_threshold_rr)

# Display results
print("Best Lambda:", best_lambda_rr)
print("Best Threshold:", best_threshold_rr)
print("Final Accuracy:", best_accuracy_rr)
print("Final F1 Score:", best_f1_rr)

[Ridge] lambda = 0.001000 | Avg f1 = 0.547281
[Ridge] lambda = 0.005000 | Avg f1 = 0.546692
[Ridge] lambda = 0.010000 | Avg f1 = 0.546010
[Ridge] lambda = 0.200000 | Avg f1 = 0.532613
=> Best lambda (Ridge): 0.001 with Avg F1 = 0.547281
Best Lambda: 0.001
Best Threshold: -0.4474975151933004
Final Accuracy: 0.8118749825900443
Final F1 Score: 0.5506802834237051


In [12]:
# Compute scores on the real test set using ridge regression weights
y_test_scores = x_test_real @ best_w_rr

# Predict labels using the optimal threshold
y_predicted = predict(y_test_scores, best_threshold_rr)

# Generate submission file for the ridge regression model
create_csv_submission(x_test_ids, y_predicted, "predRidge")

In [6]:
# Convert labels from {-1, 1} to {0, 1}
y_test=(y_test+1) // 2 
y_train=(y_train+1)//2

# Logistic Regression

In [12]:
# Test different learning rates (gamma values) for logistic regression
gamma_vals = [0.3, 0.4, 0.5]

# Select best gamma using 5-fold cross-validation
best_gamma_lr = cross_validation_logreg(y_train, X_train, k_fold=5, gammas=gamma_vals)

# Train logistic regression model with the best gamma
best_w_lr, _ = logistic_regression(
    y_train, X_train, np.zeros(X_train.shape[1]), max_iters=1000, gamma=best_gamma_lr
)

# Compute predicted probabilities on the test set
y_scores = sigmoid(X_test @ best_w_lr)

# Find optimal threshold (maximizing F1 score)
best_threshold_lr, best_f1_lr, best_accuracy_lr = optimize_thresholdlog(y_test, y_scores)

# Predict final labels
y_predicted = predict(y_scores, best_threshold_lr)

# Display performance metrics
print("Final Accuracy:", best_accuracy_lr)
print("Final F1 Score:", best_f1_lr)

gamma=0.3 | avg F1=0.550226 
gamma=0.4 | avg F1=0.549169 
gamma=0.5 | avg F1=0.550871 
=> Best gamma : 0.5 with Avg F1 = 0.550871
Final Accuracy: 0.8135045544444135
Final F1 Score: 0.5501578982731976


In [13]:
# Compute predicted probabilities on the real test set
y_test_scores = sigmoid(x_test_real @ best_w_lr)

# Predict labels using the optimal threshold
y_predicted = predict(y_test_scores, best_threshold_lr)

# Generate submission file for the logistic regression model
create_csv_submission(x_test_ids, y_predicted, "predLogisticReg")

# Reg Logistic Regression

In [ ]:
# Set regularization and learning rate values
gamma_vals = [ 0.2, 0.3, 0.4 ]
lambda_vals=[0.0001, 0.001]

# Perform 5-fold cross-validation to find the best (gamma, lambda) combination
best_lambda_reg, best_gamma_reg = cross_validate_reg_logreg(y_train,X_train,k_fold=5,lambdas=lambda_vals,gammas=gamma_vals)

# Train the final model using the best hyperparameters
best_w_reg, _ = reg_logistic_regression(y_train, X_train, best_lambda_reg,  np.zeros(X_train.shape[1]), max_iters=1000, gamma=best_gamma_reg)

# Compute predicted probabilities on the test set
y_scores = sigmoid(X_test @ best_w_reg)

# Find optimal threshold (maximizing F1 score)
best_threshold_reg, best_f1_reg, best_accuracy_reg= optimize_thresholdlog(y_test,y_scores)

# Predict final labels
y_predicted= predict(y_scores,best_threshold_reg) 

# Display performance metrics
print("Best threshhold:", best_threshold_reg)
print("Final Accuracy:", best_accuracy_reg)
print("Final F1 Score:", best_f1_reg)

λ=0.001  γ=0.2  | --- f1=0.550736
λ=0.001  γ=0.3  | --- f1=0.551637
λ=0.001  γ=0.4  | --- f1=0.549479
λ=0.01  γ=0.2  | --- f1=0.542916
λ=0.01  γ=0.3  | --- f1=0.542698
λ=0.01  γ=0.4  | --- f1=0.541957
=> Best: lambda=0.001  gamma=0.3  with avg val logloss=0.551637 
Best threshhold: 0.26567917946228264
Final Accuracy: 0.8213877823894816
Final F1 Score: 0.5462137296532201


In [15]:
# Compute predicted probabilities on the real test set
y_test_scores = sigmoid(x_test_real @ best_w_reg)

# Predict labels using the optimal threshold
y_predicted = predict(y_test_scores, best_threshold_reg)

# Generate submission file for the regularized logistic regression model
create_csv_submission(x_test_ids, y_predicted, "predRegLogisticReg")

In [16]:
# th = float(best_threshold_reg) 
# y_hat = (y_scores >= th).astype(int)

# tn = np.sum((y_test==0)&(y_hat==0))
# fp = np.sum((y_test==0)&(y_hat==1))
# fn = np.sum((y_test==1)&(y_hat==0))
# tp = np.sum((y_test==1)&(y_hat==1))
# cm = np.array([[tn, fp],
#                [fn, tp]], dtype=float)

# plt.figure(figsize=(3.1, 2.2), dpi=300)
# ax = plt.gca()

# im = ax.imshow(cm, cmap="Blues", aspect="equal",
#                norm=colors.Normalize(vmin=cm.min(), vmax=cm.max()))

# # ticks + labels
# ax.set_xticks([0,1], labels=["Pred 0", "Pred 1"])
# ax.set_yticks([0,1], labels=["True 0", "True 1"])

# thr = cm.min() + 0.55*(cm.max()-cm.min())
# for (i,j), v in np.ndenumerate(cm):
#     ax.text(j, i, f"{int(v)}",
#             ha="center", va="center",
#             color=("white" if v > thr else "black"),
#             fontsize=8)

# cbar = plt.colorbar(im, ax=ax, fraction=0.045, pad=0.04)
# cbar.ax.tick_params(labelsize=7)

# for spine in ("top","right"): ax.spines[spine].set_visible(False)
# plt.tight_layout()
# plt.savefig("figs/fig_confusion_val.pdf")
# plt.close()
# print("Saved figs/fig_confusion_val.pdf")